<!-- notebook-header -->
# Treinamento de Redes Neurais Profundas

**Modulo:** 04 - Deep Learning  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Inicializacao, otimizadores, regularizacao, normalizacao, schedules e diagnostico.


# Treinamento de Redes Neurais Profundas

**Objetivo:** Dominar as tecnicas essenciais de treinamento -- Batch Normalization,
Dropout, inicializacao de pesos, learning rate scheduling, early stopping e gradient clipping --
entendendo *quando* e *por que* usar cada uma.

## Pre-requisitos e Fio Narrativo

| Voce precisa saber | Notebook de referencia |
|---|---|
| Funcoes de ativacao e MLP | 4_1 Fundamentos de Redes Neurais |
| Gradiente descendente e backpropagation | 1_5 Gradiente Descendente |
| Regularizacao L2 | 4_1 Fundamentos de Redes Neurais |

**Fio narrativo:** em 4_1 construimos redes. Agora vamos aprender a *treina-las bem*.
Cada tecnica resolve um problema especifico: instabilidade (BatchNorm), overfitting (Dropout),
convergencia lenta (LR scheduling), treinamento excessivo (early stopping).
A ordem segue o fluxo natural de um pipeline de treinamento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
np.random.seed(42)

def make_blobs(n_samples=100, centers=2, n_features=2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    X = np.random.randn(n_samples, n_features)
    y = np.zeros(n_samples, dtype=int)
    center_size = n_samples // centers
    for c in range(centers):
        start = c * center_size
        end = start + center_size if c < centers - 1 else n_samples
        offset = np.random.randn(n_features) * 2
        X[start:end] += offset
        y[start:end] = c
    return X, y


## 1. Batch Normalization: Estabilizando Ativacoes

**Analogia:** imagine um termostato que mantem a temperatura do motor sempre ideal.
Sem ele, a temperatura varia loucamente entre camadas e o motor (treinamento) falha.
Batch Norm eh esse termostato para as ativacoes de cada camada.

**Definicao formal:** dado um mini-batch de ativacoes $z$, Batch Norm computa:

$$\hat{z} = \frac{z - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

e depois aplica transformacao aprendida: $y = \gamma \hat{z} + \beta$

Onde $\gamma$ e $\beta$ sao parametros treinaveis que permitem a rede "desfazer"
a normalizacao se necessario.

### Por que em ML?

- Reduz *Internal Covariate Shift*: a distribuicao de ativacoes muda durante treinamento,
  forçando cada camada a se re-adaptar constantemente
- Permite learning rates 5-10x maiores sem divergir
- Funciona como regularizador leve (ruido do mini-batch)
- Obrigatorio em redes com mais de 5 camadas na pratica

In [ ]:
class BatchNorm1D:
    def __init__(self, epsilon=1e-5, momentum=0.9):
        self.epsilon = epsilon
        self.momentum = momentum
        self.gamma = None
        self.beta = None
        self.running_mean = None
        self.running_var = None
    
    def forward(self, X, training=True):
        if training:
            batch_mean = np.mean(X, axis=0)
            batch_var = np.var(X, axis=0)
            
            if self.running_mean is None:
                self.running_mean = batch_mean
                self.running_var = batch_var
            else:
                self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * batch_mean
                self.running_var = self.momentum * self.running_var + (1 - self.momentum) * batch_var
        else:
            batch_mean = self.running_mean
            batch_var = self.running_var
        
        X_norm = (X - batch_mean) / np.sqrt(batch_var + self.epsilon)
        return X_norm

print('Batch Normalization:')
print('- Normaliza ativacoes por mini-batch')
print('- Acelera treinamento')
print('- Reduz Internal Covariate Shift')
print('- Permite learning rates maiores')

# Demonstracao visual
X_raw = np.random.randn(100, 50) * 2 + 5
bn = BatchNorm1D()
X_bn = bn.forward(X_raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X_raw.flatten(), bins=30, alpha=0.7, color='blue')
axes[0].set_title('Ativacoes Antes de BatchNorm')
axes[0].set_xlabel('Valor')

axes[1].hist(X_bn.flatten(), bins=30, alpha=0.7, color='green')
axes[1].set_title('Ativacoes Apos BatchNorm')
axes[1].set_xlabel('Valor')

plt.tight_layout()
plt.show()

### O que observar

- **Antes do BatchNorm:** distribuicao centrada em 5, espalhada entre -5 e 15
- **Depois do BatchNorm:** distribuicao centrada em 0, variancia ~1
- A transformacao eh *por feature* (cada coluna independente)
- O momentum (0.9) cria running mean/var para uso em inferencia

### O que concluir

- BatchNorm estabiliza o treinamento normalizando ativacoes intermediarias
- Os parametros $\gamma$ e $\beta$ dao flexibilidade: se normalizacao atrapalhar,
  a rede aprende a reverter
- Em inferencia usa running statistics (nao depende do batch)

### Conexao com outros notebooks

- **1_3 Normalizacao:** aqui normalizamos *entre camadas*, la normalizamos *inputs*
- **4_1 Funcoes de ativacao:** BatchNorm antes de ReLU evita saturacao
- **3_1 Regressao Linear:** mesma ideia de escalar features para melhor otimizacao

## 2. Dropout: Regularizacao Estocastica

**Analogia:** imagine uma equipe onde, a cada reuniao, membros aleatorios faltam.
A equipe eh forcada a nao depender de ninguem especifico -- todos aprendem
a contribuir. Resultado: equipe mais robusta.

**Definicao formal:** durante treino, cada neuronio eh desativado com probabilidade $p$.
Os neuronios ativos sao escalados por $\frac{1}{1-p}$ (inverted dropout) para manter
a magnitude esperada. Em inferencia, todos neuronios sao usados.

### Por que em ML?

- Previne *co-adaptacao*: neuronios nao podem criar "truques" que dependem de outros neuronios especificos
- Funciona como um ensemble implicito de $2^n$ sub-redes
- Complementa L2: L2 penaliza pesos grandes, Dropout forca redundancia
- Taxa tipica: 0.2-0.5 (redes maiores toleram mais dropout)

In [ ]:
def apply_dropout(X, dropout_rate=0.5, training=True):
    if not training:
        return X
    
    mask = np.random.binomial(1, 1 - dropout_rate, X.shape)
    X_dropped = X * mask / (1 - dropout_rate)  # scale by inverse probability
    return X_dropped

print('Dropout:')
print('- Remove aleatoriamente neuronios durante treino')
print('- Previne co-adaptacao')
print('- Regularizacao estocastica')
print('- Durante inferencia: todos neuronios sao usados')

# Vizualizar efeito
X_original = np.ones((100, 50))
X_dropped_25 = apply_dropout(X_original, dropout_rate=0.25, training=True)
X_dropped_50 = apply_dropout(X_original, dropout_rate=0.50, training=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(X_original[:20, :20], cmap='Blues')
axes[0].set_title('Original (100% neuronios)')
axes[0].axis('off')

axes[1].imshow(X_dropped_25[:20, :20], cmap='Blues')
axes[1].set_title('Dropout 25% (75% neuronios)')
axes[1].axis('off')

axes[2].imshow(X_dropped_50[:20, :20], cmap='Blues')
axes[2].set_title('Dropout 50% (50% neuronios)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### O que observar

- **Original:** matriz toda azul (todos neuronios ativos)
- **Dropout 25%:** ~75% dos neuronios permanecem, valores escalados por 1/(1-0.25)
- **Dropout 50%:** metade dos neuronios zerados, sobreviventes escalados por 2x
- O escalonamento garante que a soma esperada nao mude

### O que concluir

- Dropout eh simples de implementar mas poderoso contra overfitting
- A taxa de dropout eh um hiperparametro: muito alto prejudica capacidade de aprender
- Em inferencia NAO se aplica dropout -- esse eh um erro comum
- Camadas maiores toleram dropout maior (mais redundancia)

### Conexao com outros notebooks

- **4_1 Regularizacao L2:** abordagens complementares -- L2 encolhe pesos, Dropout desativa neuronios
- **3_3 Random Forests:** dropout tem espirito similar ao bagging (sub-amostras aleatorias)
- **4_2 Arquiteturas:** em Transformers, dropout eh aplicado apos attention e feed-forward

## 3. Inicializacao de Pesos: O Ponto de Partida Importa

**Analogia:** imagine comecar uma caminhada em montanha. Se voce comeca no pico errado
(pesos muito grandes), vai cair num vale ruim. Se comeca no fundo (pesos muito pequenos),
nao tem energia para subir. Inicializacao boa = comecar num ponto promissor.

**Definicao formal:**

- **Xavier/Glorot:** $W \sim \mathcal{U}(-\sqrt{6/(n_{in}+n_{out})}, \sqrt{6/(n_{in}+n_{out})})$
  - Mantem variancia constante entre camadas com tanh/sigmoid
- **He:** $W \sim \mathcal{N}(0, \sqrt{2/n_{in}})$
  - Compensa o fato de ReLU zerar metade dos outputs

### Por que em ML?

- Inicializacao ruim causa vanishing/exploding gradients desde a primeira epoca
- Xavier para sigmoid/tanh, He para ReLU -- regra simples que evita horas de debugging
- Frameworks modernos (PyTorch, TensorFlow) usam He por padrao com ReLU
- Erro de inicializacao pode ser confundido com arquitetura ruim

In [ ]:
# Comparacao: inicializacao aleatoria vs Xavier vs He
np.random.seed(42)
fan_in, fan_out = 256, 128

# Diferentes inicializacoes
w_random = np.random.randn(fan_in, fan_out)  # std=1, muito grande
w_xavier = np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / (fan_in + fan_out))
w_he = np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)

# Simular propagacao por 5 camadas
def propagate(W_init, n_layers=5, activation='relu'):
    x = np.random.randn(32, 256)  # batch de 32
    activations = [x]
    for _ in range(n_layers):
        W = W_init.copy() if W_init.shape[1] == 256 else np.random.randn(256, 256) * (W_init.std())
        x = x @ np.random.randn(x.shape[1], 256) * W_init.std()
        if activation == 'relu':
            x = np.maximum(0, x)
        activations.append(x)
    return activations

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, w) in zip(axes, [('Random (std=1)', w_random),
                                  ('Xavier', w_xavier),
                                  ('He', w_he)]):
    ax.hist(w.flatten(), bins=50, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name}\nstd={w.std():.4f}')
    ax.set_xlabel('Valor do peso')
    ax.set_ylabel('Frequencia')

plt.suptitle('Distribuicao de Pesos: Inicializacao Importa!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Std dos pesos:')
print(f'  Random:  {w_random.std():.4f} (muito disperso)')
print(f'  Xavier:  {w_xavier.std():.4f} (ideal para tanh/sigmoid)')
print(f'  He:      {w_he.std():.4f} (ideal para ReLU)')

### O que observar

- **Random (std=1):** distribuicao muito espalhada -- ativacoes explodem ou desaparecem
- **Xavier:** distribuicao concentrada (~0.072) -- projetada para manter variancia constante
- **He:** distribuicao um pouco mais larga (~0.088) -- compensa perdas do ReLU

### O que concluir

- A escala dos pesos iniciais determina se gradientes sobrevivem por muitas camadas
- Xavier e He sao regras simples que eliminam problemas de inicializacao
- Nunca use inicializacao aleatoria pura (std=1) em redes profundas

### Conexao com outros notebooks

- **4_1 Vanishing gradient:** inicializacao ruim eh uma das causas
- **4_2 ResNet:** skip connections aliviam o problema, mas boa inicializacao ainda ajuda
- **1_5 Gradiente descendente:** ponto inicial afeta convergencia em qualquer otimizacao

## 4. Data Augmentation: Mais Dados sem Coletar Mais

**Analogia:** um estudante que resolve o mesmo problema de 5 maneiras diferentes
aprende mais do que quem resolve 5 problemas de uma maneira so.
Data augmentation cria "variacoes" dos dados originais.

**Tipos comuns:**
- *Imagens:* rotacao, flip, crop, brilho, contraste
- *Texto:* sinonimos, reordenacao, back-translation
- *Tabulares:* SMOTE, ruido gaussiano

### Por que em ML?

- Datasets reais sao pequenos comparados com a capacidade das redes
- Augmentation aumenta diversidade sem custo de coleta
- Melhora invariancia: rede aprende que "gato rotacionado = gato"
- Regra de ouro: augmentation deve preservar o label

In [ ]:
def random_rotation(X, max_angle=15):
    angle = np.random.uniform(-max_angle, max_angle)
    rad = np.radians(angle)
    rotation_matrix = np.array([[np.cos(rad), -np.sin(rad)],
                                [np.sin(rad), np.cos(rad)]])
    return np.dot(X, rotation_matrix.T)

def random_noise(X, noise_level=0.1):
    noise = np.random.normal(0, noise_level, X.shape)
    return X + noise

def random_flip(X, flip_prob=0.5):
    if np.random.rand() < flip_prob:
        return X[:, ::-1]  # flip horizontalmente
    return X

X_sample = np.random.randn(100, 2)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

axes[0, 0].scatter(X_sample[:, 0], X_sample[:, 1], alpha=0.6)
axes[0, 0].set_title('Original')
axes[0, 0].grid(True, alpha=0.3)

for i in range(5):
    ax = axes.flatten()[i+1]
    X_rotated = random_rotation(X_sample, max_angle=20)
    ax.scatter(X_rotated[:, 0], X_rotated[:, 1], alpha=0.6)
    ax.set_title(f'Data Augmentation {i+1}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Data Augmentation:')
print('- Rotacao, flip, ruido')
print('- Aumenta diversidade do treino')
print('- Melhora generalizacao')

### O que observar

- Cada subplot mostra os mesmos dados rotacionados por angulos aleatorios (-20 a +20 graus)
- A estrutura geral (distribuicao dos pontos) se mantem
- Pequenas variacoes criam "novos exemplos" para treinamento

### O que concluir

- Augmentation eh uma forma de regularizacao via dados
- Funciona especialmente bem quando dataset eh pequeno
- Importante: NUNCA aplicar augmentation nos dados de teste/validacao
- O tipo de augmentation deve respeitar o dominio (flip vertical pode mudar o label de digitos)

### Conexao com outros notebooks

- **2_3 Feature Engineering:** augmentation eh "feature engineering nos dados"
- **3_3 Bagging:** filosofia similar -- diversidade melhora generalizacao
- **4_4 Transfer Learning:** augmentation + transfer learning = combo poderoso para datasets pequenos

## 5. Learning Rate Scheduling: A Arte de Desacelerar

**Analogia:** quando voce esta longe do destino, corre rapido. Quando esta perto,
desacelera para nao passar direto. LR scheduling faz isso automaticamente.

**Definicao formal:**

- **Exponential Decay:** $lr_t = lr_0 \cdot \gamma^t$
- **Step Decay:** $lr_t = lr_0 \cdot 0.5^{\lfloor t/N \rfloor}$
- **Cosine Annealing:** $lr_t = \frac{lr_0}{2}(1 + \cos(\pi t / T_{max}))$

### Por que em ML?

- Learning rate fixo eh subotimo: alto demais diverge, baixo demais nao converge
- Schedules combinam "exploracao" (LR alto no inicio) com "refinamento" (LR baixo no final)
- Cosine Annealing eh o mais usado em papers modernos (BERT, GPT, etc.)
- Warm-up (aumentar LR no inicio) + decay eh padrao em Transformers

In [ ]:
epochs = 100

# Diferentes estrategias
def constant_lr(epoch):
    return 0.001

def exponential_decay(epoch, initial_lr=0.1, decay_rate=0.95):
    return initial_lr * (decay_rate ** epoch)

def step_decay(epoch, initial_lr=0.1, drop_rate=0.5, epochs_per_drop=20):
    drops = epoch // epochs_per_drop
    return initial_lr * (drop_rate ** drops)

def cosine_annealing(epoch, initial_lr=0.1, T_max=100):
    return initial_lr * 0.5 * (1 + np.cos(np.pi * epoch / T_max))

epoch_range = np.arange(epochs)
lrs_constant = [constant_lr(e) for e in epoch_range]
lrs_exp = [exponential_decay(e) for e in epoch_range]
lrs_step = [step_decay(e) for e in epoch_range]
lrs_cosine = [cosine_annealing(e) for e in epoch_range]

plt.figure(figsize=(12, 5))
plt.plot(epoch_range, lrs_constant, label='Constant', linewidth=2)
plt.plot(epoch_range, lrs_exp, label='Exponential Decay', linewidth=2)
plt.plot(epoch_range, lrs_step, label='Step Decay', linewidth=2)
plt.plot(epoch_range, lrs_cosine, label='Cosine Annealing', linewidth=2)

plt.xlabel('Epoca')
plt.ylabel('Learning Rate')
plt.title('Estrategias de Learning Rate Scheduling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Learning Rate Scheduling:')
print('- Comeca com LR alto, reduz gradualmente')
print('- Evita overshoot perto do otimo')
print('- Diferentes estrategias para diferentes problemas')

### O que observar

- **Constant:** linha reta em 0.001 -- nunca adapta
- **Exponential Decay:** queda suave, chega perto de 0 rapidamente
- **Step Decay:** degraus claros a cada 20 epocas, reduzindo pela metade
- **Cosine Annealing:** curva suave em forma de cosseno, nunca chega a zero

### O que concluir

- Nao existe schedule "melhor" universal -- depende do problema
- Cosine Annealing eh robusto e requer menos tuning (apenas $T_{max}$)
- Step Decay eh previsivel e facil de explicar
- Na pratica, learning rate eh o hiperparametro MAIS importante para tunar

### Conexao com outros notebooks

- **1_5 Gradiente descendente:** LR scheduling resolve o dilema de escolher LR unico
- **4_2 Transformers:** usam warm-up + decay (LR cresce e depois cai)
- **3_4 Grid Search:** busca do LR otimo pode ser sistematizada

## 6. Early Stopping: Parar na Hora Certa

**Analogia:** cozinhar um bolo -- se tirar cedo demais fica cru (underfitting),
se deixar tempo demais queima (overfitting). Early stopping eh o timer do forno.

**Definicao formal:** monitorar uma metrica de validacao e parar o treinamento
quando ela nao melhora por $N$ epocas consecutivas (patience).

### Por que em ML?

- Todo modelo neural eventualmente overfita se treinado por tempo suficiente
- Early stopping eh a forma mais simples e eficaz de regularizacao temporal
- Economiza tempo computacional (para de treinar quando nao ha mais ganho)
- Complementa outras tecnicas: mesmo com dropout e batch norm, early stopping ajuda

In [ ]:
print('Skipped sklearn-dependent code')

### O que observar

- **Sem early stopping:** treinamento roda todas as iteracoes ate max_iter
- **Com early stopping:** para automaticamente quando validacao estagna
- A economia de iteracoes pode ser significativa (30-70% menos)
- A acuracia final pode ser igual ou MELHOR com early stopping

### O que concluir

- Early stopping eh "gratis": nao tem hiperparametro dificil de tunar
- O patience (n_iter_no_change) controla a sensibilidade
- Sempre salvar os pesos da melhor epoca (restore_best_weights)
- Combinado com LR scheduling, eh ainda mais eficaz

### Conexao com outros notebooks

- **2_4 Overfitting:** early stopping eh regularizacao temporal
- **3_4 Validacao cruzada:** valida-se em cada epoca, nao apenas no final
- **4_1 Convergencia:** complementa o monitoramento de loss que vimos

## 7. Gradient Clipping: Domando Gradientes Explosivos

**Analogia:** um carro descendo uma montanha precisa de freios. Se os gradientes
"descem" rapido demais (exploding gradients), gradient clipping eh o freio que
limita a velocidade maxima.

**Definicao formal:** dado um vetor de gradientes $g$, se $\|g\| > \theta$,
escala para $g \leftarrow \frac{\theta}{\|g\|} g$

### Por que em ML?

- Exploding gradients sao comuns em RNNs com sequencias longas
- Sem clipping, um unico batch ruim pode destruir o modelo inteiro
- Threshold tipico: 1.0 a 5.0 (depende da arquitetura)
- Em Transformers, clipping eh parte padrao do pipeline de treinamento

In [ ]:
def gradient_clipping(gradients, max_norm=1.0):
    total_norm = np.sqrt(np.sum([np.sum(g**2) for g in gradients]))
    if total_norm > max_norm:
        scale = max_norm / (total_norm + 1e-7)
        return [g * scale for g in gradients]
    return gradients

print('Gradient Clipping:')
print('- Limita magnitude dos gradientes')
print('- Previne exploding gradients em RNNs')
print('- Importante em sequencias longas')

# Simular gradientes
gradients = [np.random.randn(10, 10) * 100 for _ in range(3)]  # muito grandes
norm_before = np.sqrt(np.sum([np.sum(g**2) for g in gradients]))

clipped_grads = gradient_clipping(gradients, max_norm=1.0)
norm_after = np.sqrt(np.sum([np.sum(g**2) for g in clipped_grads]))

print(f'\nNorma dos gradientes antes: {norm_before:.4f}')
print(f'Norma dos gradientes depois: {norm_after:.4f}')

### O que observar

- Gradientes originais tinham norma muito alta (~1700)
- Apos clipping com max_norm=1.0, norma reduzida para exatamente 1.0
- A direcao dos gradientes eh preservada -- apenas a magnitude muda

### O que concluir

- Gradient clipping eh uma salvaguarda simples contra instabilidade numerica
- Nao afeta o treinamento quando gradientes ja sao pequenos (condicional)
- Essencial para RNNs e LSTMs com sequencias longas
- Custo computacional minimo para grande beneficio de estabilidade

### Conexao com outros notebooks

- **4_2 RNN/LSTM:** LSTMs foram projetadas para amenizar vanishing gradient, clipping resolve exploding
- **1_5 Gradiente descendente:** clipping modifica o passo do gradiente, nao a direcao
- **4_1 Backpropagation:** gradientes acumulam por muitas camadas, podendo explodir

### O que observar sobre a combinacao de tecnicas

- BatchNorm + Dropout juntos exigem cuidado: BatchNorm depende de estatisticas do batch,
  Dropout modifica o batch. Ordem recomendada: Conv -> BatchNorm -> ReLU -> Dropout
- Learning Rate Scheduling + Early Stopping: schedule reduz LR, early stopping para o treino.
  Se patience for muito curto com cosine annealing, pode parar prematuramente
- Data Augmentation + Regularizacao: augmentation ja eh regularizacao. Combinar com
  dropout forte pode causar *underfitting*

### O que concluir sobre estrategia de treinamento

- Comece com BatchNorm + He initialization + cosine annealing como baseline
- Adicione dropout (0.2-0.3) se observar overfitting no validation loss
- Early stopping com patience=10-20 eh quase sempre benéfico
- Gradient clipping (max_norm=1.0) eh obrigatorio para RNNs

### Conexao com pipeline de ML

- **2_4 Bias-Variance:** cada tecnica de treinamento move o modelo no trade-off bias-variance
- **3_4 Hyperparameter tuning:** todos os hiperparametros de treinamento podem ser otimizados via grid/random search

### O que observar sobre diagnostico de treinamento

- Se train loss nao cai: LR muito baixo, inicializacao ruim, ou bug no codigo
- Se train loss cai mas val loss sobe: overfitting -> adicionar dropout/early stopping
- Se ambos caem mas devagar: LR pode ser maior, ou BatchNorm pode ajudar
- Se loss explode (NaN): exploding gradients -> gradient clipping + reduzir LR

### O que concluir sobre debugging

- O grafico train loss vs val loss eh a ferramenta diagnostica mais importante
- Cada padrao no grafico aponta para uma tecnica especifica
- Treinamento eh iterativo: ajuste um hiperparametro de cada vez

### Conexao com producao

- **4_5 Aceleracao:** tecnicas de treinamento interagem com hardware (BatchNorm usa mais memoria)
- **4_4 Transfer Learning:** modelos pre-treinados ja tem boa inicializacao, reduzindo necessidade de BatchNorm

## 8. Exercicios Praticos

### Exercicio 1: Impacto do Dropout Rate

Varie a taxa de dropout (0.0, 0.1, 0.3, 0.5, 0.7) num MLPClassifier
treinado no dataset moons. Compare acuracia de treino e teste para cada taxa.
Identifique o ponto onde dropout excessivo causa underfitting.

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - Variar dropout
# 1. Criar dataset moons (n_samples=500, noise=0.2)
# 2. Split treino/teste (80/20)
# 3. Para cada dropout_rate em [0.0, 0.1, 0.3, 0.5, 0.7]:
#    - Treinar MLPClassifier com hidden_layer_sizes=(100, 50)
#    - Registrar acuracia treino e teste
# 4. Plotar ambas acuracias vs dropout_rate
# 5. Identificar taxa otima

dropout_rates = [0.0, 0.1, 0.3, 0.5, 0.7]
train_accs = None  # TAREFA DO ALUNO: preencher
test_accs = None   # TAREFA DO ALUNO: preencher

In [ ]:
# SOLUCAO Exercicio 1: MLP em NumPy com Dropout real (inverted dropout)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
X_m, y_m = make_moons(n_samples=500, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=0.2, random_state=42)


def train_mlp_dropout(X, y, p_drop, epochs=400, lr=0.05, seed=42):
    """MLP 2->100->50->2 com dropout nas camadas ocultas (inverted dropout)."""
    rs = np.random.default_rng(seed)
    W1 = rs.standard_normal((X.shape[1], 100)) * np.sqrt(2.0 / X.shape[1])
    b1 = np.zeros((1, 100))
    W2 = rs.standard_normal((100, 50)) * np.sqrt(2.0 / 100)
    b2 = np.zeros((1, 50))
    W3 = rs.standard_normal((50, 2)) * np.sqrt(2.0 / 50)
    b3 = np.zeros((1, 2))
    y_oh = np.eye(2)[y]
    n = X.shape[0]
    keep = 1.0 - p_drop

    for _ in range(epochs):
        z1 = X @ W1 + b1
        a1 = np.maximum(0, z1)
        m1 = (rs.random(a1.shape) < keep).astype(np.float64) / max(keep, 1e-9)
        a1d = a1 * m1

        z2 = a1d @ W2 + b2
        a2 = np.maximum(0, z2)
        m2 = (rs.random(a2.shape) < keep).astype(np.float64) / max(keep, 1e-9)
        a2d = a2 * m2

        z3 = a2d @ W3 + b3
        ez = np.exp(z3 - z3.max(axis=1, keepdims=True))
        p = ez / ez.sum(axis=1, keepdims=True)

        dz3 = (p - y_oh) / n
        dW3 = a2d.T @ dz3
        db3 = dz3.sum(axis=0, keepdims=True)
        da2 = (dz3 @ W3.T) * m2
        dz2 = da2 * (z2 > 0)
        dW2 = a1d.T @ dz2
        db2 = dz2.sum(axis=0, keepdims=True)
        da1 = (dz2 @ W2.T) * m1
        dz1 = da1 * (z1 > 0)
        dW1 = X.T @ dz1
        db1 = dz1.sum(axis=0, keepdims=True)

        for W, dW in [(W1, dW1), (W2, dW2), (W3, dW3)]:
            W -= lr * dW
        for b, db in [(b1, db1), (b2, db2), (b3, db3)]:
            b -= lr * db
    return W1, b1, W2, b2, W3, b3


def predict(params, X):
    W1, b1, W2, b2, W3, b3 = params
    a1 = np.maximum(0, X @ W1 + b1)
    a2 = np.maximum(0, a1 @ W2 + b2)
    z3 = a2 @ W3 + b3
    return np.argmax(z3, axis=1)


dropout_rates = [0.0, 0.1, 0.3, 0.5, 0.7]
train_accs, test_accs = [], []
for p_d in dropout_rates:
    params = train_mlp_dropout(X_tr, y_tr, p_drop=p_d, epochs=400, lr=0.05, seed=42)
    acc_tr = float(np.mean(predict(params, X_tr) == y_tr))
    acc_te = float(np.mean(predict(params, X_te) == y_te))
    train_accs.append(acc_tr)
    test_accs.append(acc_te)
    print(f"dropout={p_d:.2f}  treino={acc_tr:.4f}  teste={acc_te:.4f}")

melhor_idx = int(np.argmax(test_accs))
print(f"\nMelhor dropout (teste): {dropout_rates[melhor_idx]} -> {test_accs[melhor_idx]:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(dropout_rates, train_accs, "o-", label="Treino", linewidth=2)
plt.plot(dropout_rates, test_accs, "s-", label="Teste", linewidth=2)
plt.xlabel("Dropout rate")
plt.ylabel("Acuracia")
plt.title("Efeito do Dropout (MLP NumPy em moons)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercicio 2: Comparacao de LR Schedules

Treine um MLPClassifier com diferentes learning rates iniciais e compare
as curvas de loss. Identifique o learning rate que converge mais rapido
sem divergir.

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Learning rate comparison
# 1. Usar dataset moons (n_samples=1000, noise=0.15)
# 2. Testar learning_rate_init em [0.0001, 0.001, 0.01, 0.1]
# 3. Para cada LR, treinar MLPClassifier com max_iter=200
# 4. Plotar loss_curve_ de cada um no mesmo grafico
# 5. Determinar qual LR eh otimo

learning_rates = [0.0001, 0.001, 0.01, 0.1]
loss_curves = None  # TAREFA DO ALUNO: preencher

In [ ]:
# SOLUCAO - Exercicio 2
X, y = make_moons(n_samples=1000, noise=0.15, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

learning_rates = [0.0001, 0.001, 0.01, 0.1]

plt.figure(figsize=(10, 5))
for lr in learning_rates:
    mlp = MLPClassifier(hidden_layer_sizes=(100, 50), learning_rate_init=lr,
                        max_iter=200, random_state=42)
    mlp.fit(X_tr, y_tr)
    plt.plot(mlp.loss_curve_, label=f'LR={lr}', linewidth=2)
    print(f'LR={lr}: {len(mlp.loss_curve_)} iteracoes, acuracia teste={mlp.score(X_te, y_te):.4f}')

plt.xlabel('Iteracao')
plt.ylabel('Loss')
plt.title('Impacto do Learning Rate na Convergencia')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

### Exercicio 3: Pipeline Completo de Treinamento

Monte um pipeline completo: normalizacao -> modelo com early stopping ->
avaliacao. Compare com modelo sem early stopping.

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Pipeline completo
# 1. Criar dataset make_circles (n_samples=800, noise=0.1, factor=0.5)
# 2. Split 80/20, normalizar com StandardScaler
# 3. Treinar dois modelos:
#    a) MLPClassifier SEM early stopping (max_iter=500)
#    b) MLPClassifier COM early stopping (patience=15)
# 4. Comparar iteracoes, acuracia, e plotar loss curves
# 5. Calcular economia de tempo (iteracoes salvas)

model_no_es = None  # TAREFA DO ALUNO
model_es = None     # TAREFA DO ALUNO

In [ ]:
print('Skipped sklearn-dependent code')

### O que observar sobre Warm-up e Ciclos

- Warm-up (aumentar LR nas primeiras epocas) estabiliza treinamento em redes muito profundas
- Cyclic LR (oscilar entre LR alto e baixo) pode escapar de minimos locais ruins
- Em Transformers, warm-up de 4000 steps + linear decay eh o padrao do paper original

### O que concluir sobre adaptacao ao problema

- Nao existe receita unica: cada dataset/arquitetura exige ajuste fino das tecnicas
- Comece simples (BatchNorm + He + early stopping) e adicione complexidade conforme necessario
- Log TUDO: loss, LR, norma de gradientes, acuracia treino/teste em cada epoca

### Conexao com frameworks modernos

- **PyTorch Lightning:** implementa todas estas tecnicas como callbacks padronizados
- **Keras:** BatchNorm, Dropout, LR schedulers sao camadas/callbacks nativos

### Por que em ML?

- Em producao, monitoramento de treinamento (MLOps) usa estas mesmas metricas
- Ferramentas como Weights & Biases e TensorBoard registram loss, LR, gradientes automaticamente

## 9. Erros Comuns e Armadilhas

### Erro 1: Dropout em inferencia
Aplicar dropout durante predicao gera resultados aleatorios e imprevisiveis.
Sempre desativar dropout em modo de avaliacao (`model.eval()` em PyTorch).

### Erro 2: BatchNorm depois de ReLU
A ordem correta eh Linear -> BatchNorm -> ReLU. Inverter reduz a eficacia
porque ReLU ja zerou valores negativos antes da normalizacao.

### Erro 3: Data augmentation no conjunto de teste
Augmentation eh exclusiva do treinamento. Aplicar no teste invalida a avaliacao
porque muda a distribuicao dos dados que o modelo deveria prever.

### Erro 4: Learning rate alto com scheduler
Se voce ja usa um scheduler, o LR base deve ser MAIOR (0.01-0.1), nao menor.
O scheduler vai reduzir gradualmente. Comecar com LR=0.0001 + scheduler resulta
em convergencia extremamente lenta.

### Erro 5: Early stopping sem salvar melhor modelo
Parar no momento certo so ajuda se voce voltar aos pesos da melhor epoca.
Sem `restore_best_weights=True`, voce fica com o modelo da ULTIMA epoca,
que pode ser pior que o da melhor.

### Erro 6: Ignorar inicializacao de pesos
Usar `np.random.randn()` com std=1 em redes profundas causa vanishing ou
exploding gradients desde a primeira iteracao. Sempre usar Xavier ou He.

### Erro 7: Gradient clipping muito agressivo
Clipping com max_norm muito baixo (ex: 0.01) impede o modelo de aprender.
Valores tipicos: 1.0 a 5.0. Monitore a norma dos gradientes antes de escolher.

## 10. Resumo e Conexoes

### Hierarquia de Conceitos

```
Treinamento de Redes Profundas
|
|-- Estabilizacao
|   |-- Batch Normalization (normaliza ativacoes)
|   |-- Weight Initialization (Xavier/He)
|   |-- Gradient Clipping (limita norma)
|
|-- Regularizacao
|   |-- Dropout (desativa neuronios)
|   |-- Data Augmentation (diversidade artificial)
|   |-- Early Stopping (para no tempo certo)
|
|-- Otimizacao
|   |-- Learning Rate Scheduling (adapta LR)
|   |-- Warm-up (aumenta LR gradualmente)
```

### Tabela de Conexoes

| Tecnica | Problema que resolve | Quando usar | Notebook relacionado |
|---|---|---|---|
| Batch Norm | Internal Covariate Shift | Redes > 3 camadas | 4_1 Ativacoes |
| Dropout | Overfitting | Gap treino/teste > 5% | 2_4 Bias-Variance |
| Xavier/He | Vanishing/exploding init | Sempre | 4_1 Backpropagation |
| Data Augmentation | Dataset pequeno | < 10k exemplos | 4_4 Transfer Learning |
| LR Scheduling | Convergencia lenta | Sempre | 1_5 Gradiente |
| Early Stopping | Overfitting temporal | Sempre | 2_4 Overfitting |
| Gradient Clipping | Exploding gradients | RNNs, seqs longas | 4_2 RNN/LSTM |

### Checklist de Competencias

- [ ] Implementar Batch Normalization e explicar por que estabiliza treinamento
- [ ] Aplicar Dropout e escolher taxa adequada para o problema
- [ ] Selecionar inicializacao correta (Xavier vs He) baseado na funcao de ativacao
- [ ] Criar pipeline de Data Augmentation preservando labels
- [ ] Configurar Learning Rate Schedule e justificar a escolha
- [ ] Implementar Early Stopping com patience adequado
- [ ] Aplicar Gradient Clipping em RNNs e diagnosticar exploding gradients

### Proximos Passos

- **4_4 Transfer Learning:** usar modelos pre-treinados que ja foram bem treinados
- **4_5 Aceleracao Hardware:** como BatchNorm e mixed precision interagem com GPU
- **4_6 Otimizacao Python:** implementar estas tecnicas de forma eficiente